In [1]:
# ============================================================
# Cell 1. Install packages
# ============================================================

%pip install -q pandas numpy requests tqdm python-dotenv openpyxl lxml beautifulsoup4 statsmodels scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Cell 2. Configuration
# ============================================================

from pathlib import Path
from getpass import getpass
import os
import json
import time
import zipfile
import re
import io
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv, set_key

import statsmodels.formula.api as smf
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project folders
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"
OUT_DIR = PROJECT_DIR / "output_governance_direction"
LOG_DIR = PROJECT_DIR / "logs"

for d in [DATA_DIR, RAW_DIR, CACHE_DIR, OUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# User KRX file
# ------------------------------------------------------------

KRX_FILE = Path(r"C:\Users\starw\Downloads\krx_market_list_clean.csv")

if not KRX_FILE.exists():
    raise FileNotFoundError(f"KRX file not found: {KRX_FILE}")

# ------------------------------------------------------------
# Analysis period
# ------------------------------------------------------------

START_YEAR = 2018
END_YEAR = 2024
YEARS = list(range(START_YEAR, END_YEAR + 1))

# 사업보고서
REPORT_CODE = "11011"

# 전체 표본으로 진행
PILOT_MODE = False
PILOT_N_FIRMS = 50

# API request settings
REQUEST_SLEEP = 0.20
TIMEOUT = 40
MAX_RETRIES = 3

# ------------------------------------------------------------
# Open DART API key
# ------------------------------------------------------------

ENV_PATH = PROJECT_DIR / ".env"
load_dotenv(ENV_PATH)

DART_API_KEY = os.getenv("DART_API_KEY")

if not DART_API_KEY:
    DART_API_KEY = getpass("Open DART API key를 입력하세요: ").strip()
    set_key(str(ENV_PATH), "DART_API_KEY", DART_API_KEY)
    print(f"API key saved to {ENV_PATH}")
else:
    print("DART_API_KEY loaded from .env")

BASE_URL = "https://opendart.fss.or.kr/api"

print("PROJECT_DIR:", PROJECT_DIR)
print("KRX_FILE:", KRX_FILE)
print("YEARS:", YEARS)
print("PILOT_MODE:", PILOT_MODE)

DART_API_KEY loaded from .env
PROJECT_DIR: c:\Users\starw\.vscode\practice
KRX_FILE: C:\Users\starw\Downloads\krx_market_list_clean.csv
YEARS: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
PILOT_MODE: False


In [3]:
# ============================================================
# Cell 3. Utility functions
# ============================================================

def clean_stock_code(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    s = re.sub(r"[^0-9]", "", s)
    if s == "":
        return np.nan
    return s.zfill(6)


def is_standard_stock_code(x):
    return bool(re.fullmatch(r"\d{6}", str(x).strip()))


def parse_number(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).strip()
    
    if s in ["", "-", "—", "–", "해당사항 없음", "해당사항없음", "nan", "None"]:
        return np.nan
    
    if "해당" in s and "없" in s:
        return np.nan
    
    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1]
    
    s = s.replace(",", "")
    matches = re.findall(r"-?\d+\.?\d*", s)
    
    if not matches:
        return np.nan
    
    val = float(matches[0])
    if neg:
        val = -abs(val)
    return val


def safe_divide(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b


def winsorize_series(s, lower=0.01, upper=0.99):
    s = s.copy()
    if s.dropna().empty:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def cache_path(endpoint_name, params):
    key_parts = [endpoint_name]
    for k in sorted(params.keys()):
        if k == "crtfc_key":
            continue
        key_parts.append(f"{k}-{params[k]}")
    fname = "__".join(key_parts)
    fname = re.sub(r"[^A-Za-z0-9가-힣_\-\.]", "_", fname)
    return CACHE_DIR / f"{fname}.json"


def dart_get_json(endpoint_name, params, use_cache=True):
    params = dict(params)
    params["crtfc_key"] = DART_API_KEY
    
    cp = cache_path(endpoint_name, params)
    if use_cache and cp.exists():
        with open(cp, "r", encoding="utf-8") as f:
            return json.load(f)
    
    url = f"{BASE_URL}/{endpoint_name}"
    last_error = None
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(REQUEST_SLEEP)
            r = requests.get(url, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()
            
            if use_cache:
                with open(cp, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
            return data
        
        except Exception as e:
            last_error = str(e)
            time.sleep(attempt)
    
    return {
        "status": "ERROR",
        "message": last_error,
        "list": []
    }


def dart_list(endpoint_name, params, use_cache=True):
    data = dart_get_json(endpoint_name, params, use_cache=use_cache)
    status = str(data.get("status", ""))
    
    if status == "000":
        return data.get("list", [])
    elif status == "013":
        return []
    else:
        return []


def save_df(df, filename):
    path = OUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path} | shape={df.shape}")
    return path


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def print_basic(df, name, n=5):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head(n))

In [4]:
# ============================================================
# Cell 4. Load KRX sample
# ============================================================

krx = pd.read_csv(KRX_FILE, dtype=str, encoding="utf-8-sig")

print_basic(krx, "KRX raw")

required = ["stock_code", "corp_name", "market", "sector"]
missing = [c for c in required if c not in krx.columns]

if missing:
    raise ValueError(f"필수 컬럼 없음: {missing}. 현재 컬럼: {krx.columns.tolist()}")

krx["stock_code"] = krx["stock_code"].apply(clean_stock_code)
krx["market"] = krx["market"].astype(str).str.strip()
krx["sector"] = krx["sector"].astype(str).str.strip()

financial_keywords = [
    "금융", "은행", "보험", "증권", "투자", "신탁",
    "카드", "캐피탈", "리스", "부동산", "회사 본부", "기금"
]

if "is_financial_conservative" in krx.columns:
    krx["is_financial_conservative"] = krx["is_financial_conservative"].astype(str).str.lower().map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
    krx["is_financial_conservative"] = krx["is_financial_conservative"].fillna(
        krx["sector"].apply(lambda x: any(k in x for k in financial_keywords))
    )
else:
    krx["is_financial_conservative"] = krx["sector"].apply(
        lambda x: any(k in x for k in financial_keywords)
    )

sample = krx[
    (krx["market"] == "KOSPI") &
    (krx["stock_code"].apply(is_standard_stock_code)) &
    (~krx["is_financial_conservative"])
].copy()

sample = sample.drop_duplicates("stock_code").reset_index(drop=True)

print("Full KOSPI nonfinancial sample:", sample.shape)

if PILOT_MODE:
    sample = sample.head(PILOT_N_FIRMS).copy()
    print(f"PILOT MODE: {len(sample)} firms")

display(sample[["stock_code", "corp_name", "market", "sector"]].head(20))

save_df(sample, "01_sample_kospi_nonfinancial.csv")


KRX raw
shape: (2764, 12)
columns: ['stock_code', 'corp_name', 'market', 'market_raw', 'sector', 'main_products', 'listing_date', 'fiscal_month', 'region', 'is_financial_narrow', 'is_financial_conservative', 'exclude_reason']


,stock_code,corp_name,market,market_raw,sector,main_products,listing_date,fiscal_month,region,is_financial_narrow,is_financial_conservative,exclude_reason
0,477850,마키나락스,KOSDAQ,코스닥,소프트웨어 개발 및 공급업,Runway Platform (산업특화 인공지능 개발 및 운영 체계 구축 플랫폼),2026-05-20,12월,서울특별시,False,False,NaN
1,288180,케이피항공산업,KOSDAQ,코스닥,"항공기,우주선 및 부품 제조업",항공기 및 우주 방산 구조물 제작,2026-05-19,12월,경상남도,False,False,NaN
2,487580,폴레드,KOSDAQ,코스닥,가정용 기기 제조업,"유아용품(카시트 및 관련 악세서리, 유아가전 등)",2026-05-14,12월,충청남도,False,False,NaN
3,439960,코스모로보틱스,KOSDAQ,코스닥,의료용 기기 제조업,웨어러블 로봇,2026-05-11,12월,서울특별시,False,False,NaN
4,0129K0,신한제18호스팩,KOSDAQ,코스닥,금융 지원 서비스업,금융 지원 서비스업,2026-04-30,12월,서울특별시,True,True,금융


Full KOSPI nonfinancial sample: (692, 12)


,stock_code,corp_name,market,sector
0,217590,티엠씨,KOSPI,절연선 및 케이블 제조업
1,001200,삼양바이오팜,KOSPI,의약품 제조업
2,317450,명인제약,KOSPI,의약품 제조업
3,439260,대한조선,KOSPI,선박 및 보트 건조업
4,483650,달바글로벌,KOSPI,기타 화학제품 제조업
5,480370,씨케이솔루션,KOSPI,일반 목적용 기계 제조업
6,064400,LG씨엔에스,KOSPI,"컴퓨터 프로그래밍, 시스템 통합 및 관리업"
7,484870,엠앤씨솔루션,KOSPI,특수 목적용 기계 제조업
8,475560,더본코리아,KOSPI,상품 종합 도매업
9,489790,한화비전,KOSPI,통신 및 방송 장비 제조업


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\01_sample_kospi_nonfinancial.csv | shape=(692, 12)


WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/01_sample_kospi_nonfinancial.csv')

In [5]:
# ============================================================
# Cell 5. Download and parse corp codes
# ============================================================

def download_corp_codes():
    zip_path = RAW_DIR / "corpCode.zip"
    xml_path = RAW_DIR / "CORPCODE.xml"
    
    if xml_path.exists():
        return xml_path
    
    url = f"{BASE_URL}/corpCode.xml"
    params = {"crtfc_key": DART_API_KEY}
    
    print("Downloading corpCode.xml...")
    r = requests.get(url, params=params, timeout=TIMEOUT)
    r.raise_for_status()
    
    with open(zip_path, "wb") as f:
        f.write(r.content)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(RAW_DIR)
    
    if not xml_path.exists():
        candidates = list(RAW_DIR.glob("*.xml"))
        if candidates:
            xml_path = candidates[0]
        else:
            raise FileNotFoundError("CORPCODE.xml not found.")
    
    return xml_path


def parse_corp_codes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    rows = []
    
    for item in root.findall("list"):
        row = {}
        for child in item:
            row[child.tag] = child.text
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df["stock_code"] = df["stock_code"].apply(clean_stock_code)
    return df


corp_xml = download_corp_codes()
corp_codes = parse_corp_codes(corp_xml)

corp_listed = corp_codes[
    corp_codes["stock_code"].apply(is_standard_stock_code)
].copy()

sample_dart = sample.merge(
    corp_listed[["corp_code", "corp_name", "stock_code", "modify_date"]],
    on="stock_code",
    how="left",
    suffixes=("", "_dart")
)

print_basic(sample_dart, "sample_dart")
print("Missing corp_code:", sample_dart["corp_code"].isna().sum())

sample_dart = sample_dart.dropna(subset=["corp_code"]).copy()
sample_dart["corp_code"] = sample_dart["corp_code"].astype(str).str.zfill(8)

save_df(sample_dart, "02_sample_with_corp_code.csv")


sample_dart
shape: (711, 15)
columns: ['stock_code', 'corp_name', 'market', 'market_raw', 'sector', 'main_products', 'listing_date', 'fiscal_month', 'region', 'is_financial_narrow', 'is_financial_conservative', 'exclude_reason', 'corp_code', 'corp_name_dart', 'modify_date']


,stock_code,corp_name,market,market_raw,sector,main_products,listing_date,fiscal_month,region,is_financial_narrow,is_financial_conservative,exclude_reason,corp_code,corp_name_dart,modify_date
0,217590,티엠씨,KOSPI,유가,절연선 및 케이블 제조업,"선박용케이블, 해양용케이블, 전력용케이블, 광케이블 등",2025-12-15,12월,충청남도,False,False,NaN,00949161,티엠씨,20251215
1,001200,삼양바이오팜,KOSPI,유가,의약품 제조업,의료기기 및 의약품,2025-11-24,12월,경기도,False,False,NaN,00131054,유진증권,20230110
2,001200,삼양바이오팜,KOSPI,유가,의약품 제조업,의료기기 및 의약품,2025-11-24,12월,경기도,False,False,NaN,01965333,삼양바이오팜,20251124
3,317450,명인제약,KOSPI,유가,의약품 제조업,"신경정신계열, 구강계열 완제의약품",2025-10-01,12월,경기도,False,False,NaN,00179489,명인제약,20260327
4,439260,대한조선,KOSPI,유가,선박 및 보트 건조업,선박 및 보트 건조업,2025-08-01,12월,전라남도,False,False,NaN,00182696,대한조선,20260420


Missing corp_code: 0
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\02_sample_with_corp_code.csv | shape=(711, 15)


WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/02_sample_with_corp_code.csv')

In [6]:
# ============================================================
# Cell 6. Collect filings and correction variables
# ============================================================

def fetch_filings(corp_code, year):
    rows = []
    params = {
        "corp_code": corp_code,
        "bgn_de": f"{year}0101",
        "end_de": f"{year}1231",
        "page_no": 1,
        "page_count": 100,
        "pblntf_ty": "A"
    }
    
    while True:
        data = dart_get_json("list.json", params)
        status = str(data.get("status", ""))
        
        if status == "000":
            rows.extend(data.get("list", []))
            total_page = int(data.get("total_page", 1) or 1)
            if int(params["page_no"]) >= total_page:
                break
            params["page_no"] += 1
        
        elif status == "013":
            break
        
        else:
            break
    
    for r in rows:
        r["query_year"] = year
    return rows


filing_rows = []

for _, firm in tqdm(sample_dart.iterrows(), total=len(sample_dart), desc="filings"):
    for year in YEARS:
        filing_rows.extend(fetch_filings(firm["corp_code"], year))

filings = pd.DataFrame(filing_rows)

if filings.empty:
    raise ValueError("No filings collected. API key or corp_code matching may be wrong.")

filings["year"] = filings["query_year"].astype(int)
filings["report_nm"] = filings["report_nm"].astype(str)
filings["rcept_dt"] = filings["rcept_dt"].astype(str)

annual_filings = filings[
    filings["report_nm"].str.contains("사업보고서", na=False)
].copy()

annual_filings["CorrectionDummy_filing"] = annual_filings["report_nm"].str.contains("정정", na=False).astype(int)

annual_corrections = annual_filings[
    annual_filings["CorrectionDummy_filing"] == 1
].copy()

correction_agg = (
    annual_corrections
    .groupby(["corp_code", "year"], as_index=False)
    .agg(
        CorrectionCount=("rcept_no", "nunique"),
        FirstCorrectionDate=("rcept_dt", "min"),
        LastCorrectionDate=("rcept_dt", "max")
    )
)

base_panel = (
    sample_dart[["stock_code", "corp_code", "corp_name", "market", "sector", "listing_date"]]
    .assign(_key=1)
    .merge(pd.DataFrame({"year": YEARS, "_key": 1}), on="_key")
    .drop(columns="_key")
)

panel = base_panel.merge(correction_agg, on=["corp_code", "year"], how="left")
panel["CorrectionCount"] = panel["CorrectionCount"].fillna(0).astype(int)
panel["CorrectionDummy"] = (panel["CorrectionCount"] > 0).astype(int)

print_basic(filings, "filings")
print_basic(annual_filings, "annual_filings")
print_basic(annual_corrections, "annual_corrections")
print(panel["CorrectionDummy"].value_counts())

save_df(filings, "03_all_filings.csv")
save_df(annual_filings, "04_annual_filings.csv")
save_df(annual_corrections, "05_annual_corrections.csv")
save_df(panel, "06_panel_correction.csv")

filings:   0%|          | 0/711 [00:00<?, ?it/s]


filings
shape: (20774, 11)
columns: ['corp_code', 'corp_name', 'stock_code', 'corp_cls', 'report_nm', 'rcept_no', 'flr_nm', 'rcept_dt', 'rm', 'query_year', 'year']


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm,query_year,year
0,00131054,유진증권,001200,Y,[첨부정정]분기보고서 (2018.09),20181115000123,유진증권,20181115,,2018,2018
1,00131054,유진증권,001200,Y,분기보고서 (2018.09),20181114001825,유진증권,20181114,정,2018,2018
2,00131054,유진증권,001200,Y,반기보고서 (2018.06),20180814001492,유진증권,20180814,,2018,2018
3,00131054,유진증권,001200,Y,분기보고서 (2018.03),20180511004158,유진증권,20180511,,2018,2018
4,00131054,유진증권,001200,Y,사업보고서 (2017.12),20180402003641,유진증권,20180402,연,2018,2018



annual_filings
shape: (6057, 12)
columns: ['corp_code', 'corp_name', 'stock_code', 'corp_cls', 'report_nm', 'rcept_no', 'flr_nm', 'rcept_dt', 'rm', 'query_year', 'year', 'CorrectionDummy_filing']


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm,query_year,year,CorrectionDummy_filing
4,00131054,유진증권,001200,Y,사업보고서 (2017.12),20180402003641,유진증권,20180402,연,2018,2018,0
8,00131054,유진증권,001200,Y,사업보고서 (2018.12),20190401002982,유진증권,20190401,연,2019,2019,0
12,00131054,유진증권,001200,Y,사업보고서 (2019.12),20200330002456,유진증권,20200330,연,2020,2020,0
16,00131054,유진증권,001200,Y,사업보고서 (2020.12),20210317000664,유진증권,20210317,연,2021,2021,0
20,00131054,유진증권,001200,Y,사업보고서 (2021.12),20220316000791,유진증권,20220316,연,2022,2022,0



annual_corrections
shape: (1465, 12)
columns: ['corp_code', 'corp_name', 'stock_code', 'corp_cls', 'report_nm', 'rcept_no', 'flr_nm', 'rcept_dt', 'rm', 'query_year', 'year', 'CorrectionDummy_filing']


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm,query_year,year,CorrectionDummy_filing
50,00139834,LG씨엔에스,064400,Y,[기재정정]사업보고서 (2021.12),20220405002309,LG씨엔에스,20220405,연,2022,2022,1
80,00171265,파라다이스,034230,Y,[기재정정]사업보고서 (2020.12),20210419000082,파라다이스,20210419,연,2021,2021,1
104,01190568,에이피알,278470,Y,[기재정정]사업보고서 (2017.12),20180528000239,에이피알,20180528,정,2018,2018,1
106,01190568,에이피알,278470,Y,[기재정정]사업보고서 (2017.12),20180417000178,에이피알,20180417,정,2018,2018,1
111,01190568,에이피알,278470,Y,[기재정정]사업보고서 (2018.12),20191114002071,에이피알,20191114,정,2019,2019,1


CorrectionDummy
0    3985
1     992
Name: count, dtype: int64
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\03_all_filings.csv | shape=(20774, 11)
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\04_annual_filings.csv | shape=(6057, 12)
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\05_annual_corrections.csv | shape=(1465, 12)
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\06_panel_correction.csv | shape=(4977, 11)


WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/06_panel_correction.csv')

In [7]:
# ============================================================
# Cell 7. Classify material corrections
# ============================================================

MATERIAL_KEYWORDS = [
    "재무제표", "연결재무제표", "재무상태표", "손익계산서", "포괄손익계산서",
    "현금흐름표", "자본변동표",
    "매출", "영업이익", "당기순이익", "자산총계", "부채총계", "자본총계",
    "감사의견", "감사인", "회계감사", "감사용역", "비감사용역",
    "내부회계관리제도",
    "최대주주", "소액주주", "사외이사", "이사회", "감사위원",
    "특수관계", "종속기업", "관계기업", "타법인", "출자"
]

def document_text_cache(rcept_no):
    return CACHE_DIR / f"document_text_{rcept_no}.txt"


def fetch_document_text(rcept_no):
    cp = document_text_cache(rcept_no)
    if cp.exists():
        return cp.read_text(encoding="utf-8", errors="ignore")
    
    url = f"{BASE_URL}/document.xml"
    params = {"crtfc_key": DART_API_KEY, "rcept_no": rcept_no}
    
    try:
        time.sleep(REQUEST_SLEEP)
        r = requests.get(url, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        content = r.content
        
        text = ""
        if content[:2] == b"PK":
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                for name in z.namelist():
                    raw = z.read(name)
                    for enc in ["utf-8", "cp949", "euc-kr"]:
                        try:
                            text += raw.decode(enc, errors="ignore")
                            break
                        except:
                            pass
        else:
            for enc in ["utf-8", "cp949", "euc-kr"]:
                try:
                    text = content.decode(enc, errors="ignore")
                    break
                except:
                    pass
        
        cp.write_text(text, encoding="utf-8", errors="ignore")
        return text
    
    except Exception:
        return ""


material_rows = []

for _, r in tqdm(annual_corrections.iterrows(), total=len(annual_corrections), desc="material correction"):
    text = fetch_document_text(r["rcept_no"])
    hits = sorted(set([kw for kw in MATERIAL_KEYWORDS if kw in text]))
    
    material_rows.append({
        "corp_code": r["corp_code"],
        "corp_name": r.get("corp_name"),
        "year": int(r["year"]),
        "rcept_no": r["rcept_no"],
        "rcept_dt": r["rcept_dt"],
        "report_nm": r["report_nm"],
        "MaterialCorrection": int(len(hits) > 0),
        "MaterialKeywordHits": len(hits),
        "MaterialKeywords": "|".join(hits)
    })

material_detail = pd.DataFrame(material_rows)

if material_detail.empty:
    material_agg = pd.DataFrame(columns=[
        "corp_code", "year", "MaterialCorrectionCount", "MaterialKeywordHits"
    ])
else:
    material_agg = (
        material_detail
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            MaterialCorrectionCount=("MaterialCorrection", "sum"),
            MaterialKeywordHits=("MaterialKeywordHits", "sum")
        )
    )

panel = panel.merge(material_agg, on=["corp_code", "year"], how="left")
panel["MaterialCorrectionCount"] = panel["MaterialCorrectionCount"].fillna(0).astype(int)
panel["MaterialCorrectionDummy"] = (panel["MaterialCorrectionCount"] > 0).astype(int)
panel["MaterialKeywordHits"] = panel["MaterialKeywordHits"].fillna(0).astype(int)

print_basic(material_detail, "material_detail")
print(panel[["CorrectionDummy", "CorrectionCount", "MaterialCorrectionDummy", "MaterialCorrectionCount"]].describe())

save_df(material_detail, "07_material_corrections_detail.csv")
save_df(panel, "08_panel_material_correction.csv")

material correction:   0%|          | 0/1465 [00:00<?, ?it/s]


material_detail
shape: (1465, 9)
columns: ['corp_code', 'corp_name', 'year', 'rcept_no', 'rcept_dt', 'report_nm', 'MaterialCorrection', 'MaterialKeywordHits', 'MaterialKeywords']


,corp_code,corp_name,year,rcept_no,rcept_dt,report_nm,MaterialCorrection,MaterialKeywordHits,MaterialKeywords
0,00139834,LG씨엔에스,2022,20220405002309,20220405,[기재정정]사업보고서 (2021.12),0,0,
1,00171265,파라다이스,2021,20210419000082,20210419,[기재정정]사업보고서 (2020.12),0,0,
2,01190568,에이피알,2018,20180528000239,20180528,[기재정정]사업보고서 (2017.12),0,0,
3,01190568,에이피알,2018,20180417000178,20180417,[기재정정]사업보고서 (2017.12),0,0,
4,01190568,에이피알,2019,20191114002071,20191114,[기재정정]사업보고서 (2018.12),0,0,


       CorrectionDummy  CorrectionCount  MaterialCorrectionDummy  \
count      4977.000000      4977.000000              4977.000000   
mean          0.199317         0.294354                 0.069520   
std           0.399527         0.826580                 0.254362   
min           0.000000         0.000000                 0.000000   
25%           0.000000         0.000000                 0.000000   
50%           0.000000         0.000000                 0.000000   
75%           0.000000         0.000000                 0.000000   
max           1.000000        13.000000                 1.000000   

       MaterialCorrectionCount  
count              4977.000000  
mean                  0.093631  
std                   0.454605  
min                   0.000000  
25%                   0.000000  
50%                   0.000000  
75%                   0.000000  
max                  10.000000  
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\07_material_corrections

WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/08_panel_material_correction.csv')

In [8]:
# ============================================================
# Cell 8. Collect periodic report key info
# ============================================================

ENDPOINTS = {
    "auditor_opinion": "accnutAdtorNmNdAdtOpinion.json",
    "audit_contract": "adtServcCnclsSttus.json",
    "non_audit_contract": "accnutAdtorNonAdtServcCnclsSttus.json",
    "outside_director": "outcmpnyDrctrNdChangeSttus.json",
    "largest_shareholder": "hyslrSttus.json",
    "minority_shareholder": "mrhlSttus.json",
}

def fetch_endpoint(endpoint, corp_code, year):
    params = {
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": REPORT_CODE
    }
    return dart_list(endpoint, params)

endpoint_frames = {}

for label, endpoint in ENDPOINTS.items():
    rows = []
    print("\nCollecting:", label, endpoint)
    
    for _, firm in tqdm(sample_dart.iterrows(), total=len(sample_dart), desc=label):
        for year in YEARS:
            out = fetch_endpoint(endpoint, firm["corp_code"], year)
            for item in out:
                item["query_year"] = year
                item["endpoint_label"] = label
                rows.append(item)
    
    df = pd.DataFrame(rows)
    endpoint_frames[label] = df
    save_df(df, f"raw_{label}.csv")
    print_basic(df, f"raw_{label}", n=3)


Collecting: auditor_opinion accnutAdtorNmNdAdtOpinion.json


auditor_opinion:   0%|          | 0/711 [00:00<?, ?it/s]

Saved: c:\Users\starw\.vscode\practice\output_governance_direction\raw_auditor_opinion.csv | shape=(15949, 13)

raw_auditor_opinion
shape: (15949, 13)
columns: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'bsns_year', 'adtor', 'adt_opinion', 'adt_reprt_spcmnt_matter', 'stlm_dt', 'query_year', 'endpoint_label', 'emphs_matter', 'core_adt_matter']


,rcept_no,corp_cls,corp_code,corp_name,bsns_year,adtor,adt_opinion,adt_reprt_spcmnt_matter,stlm_dt,query_year,endpoint_label,emphs_matter,core_adt_matter
0,20190401002982,Y,00131054,유진증권,제66기,삼정회계법인\n대표이사 김 교 태,연결 : 적정\n별도 : 적정,연결 및 별도\n해당 사항 없음,2018-12-31,2018,auditor_opinion,NaN,NaN
1,20190401002982,Y,00131054,유진증권,제65기,삼정회계법인\n대표이사 김 교 태,연결 : 적정\n별도 : 적정,연결 및 별도\n해당 사항 없음,2018-12-31,2018,auditor_opinion,NaN,NaN
2,20190401002982,Y,00131054,유진증권,제64기,삼정회계법인\n대표이사 김 교 태,연결 : 적정\n별도 : 적정,연결 및 별도\n해당 사항 없음,2018-12-31,2018,auditor_opinion,NaN,NaN



Collecting: audit_contract adtServcCnclsSttus.json


...

,rcept_no,corp_cls,corp_code,corp_name,stock_knd,nm,relate,bsis_posesn_stock_co,bsis_posesn_stock_qota_rt,trmend_posesn_stock_co,trmend_posesn_stock_qota_rt,rm,stlm_dt,query_year,endpoint_label
0,20190401002982,Y,00131054,유진증권,보통주,유진기업(주),본인,"26,400,920",27.25,"26,400,920",27.25,-,2018-12-31,2018,largest_shareholder
1,20190401002982,Y,00131054,유진증권,보통주,유창수,임원,"564,707",0.58,"764,707",0.79,-,2018-12-31,2018,largest_shareholder
2,20190401002982,Y,00131054,유진증권,보통주,유재필,계열사임원,"165,884",0.17,"165,884",0.17,-,2018-12-31,2018,largest_shareholder



Collecting: minority_shareholder mrhlSttus.json


minority_shareholder:   0%|          | 0/711 [00:00<?, ?it/s]

Saved: c:\Users\starw\.vscode\practice\output_governance_direction\raw_minority_shareholder.csv | shape=(4635, 14)

raw_minority_shareholder
shape: (4635, 14)
columns: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'se', 'shrholdr_co', 'shrholdr_tot_co', 'shrholdr_rate', 'hold_stock_co', 'stock_tot_co', 'hold_stock_rate', 'stlm_dt', 'query_year', 'endpoint_label']


,rcept_no,corp_cls,corp_code,corp_name,se,shrholdr_co,shrholdr_tot_co,shrholdr_rate,hold_stock_co,stock_tot_co,hold_stock_rate,stlm_dt,query_year,endpoint_label
0,20190401002982,Y,00131054,유진증권,소액주주,"38,356",-,99.97%,"67,052,361",-,69.21%,2018-12-31,2018,minority_shareholder
1,20200330002456,Y,00131054,유진증권,소액주주,"36,199",-,99.98%,"68,967,787",-,71.20%,2019-12-31,2019,minority_shareholder
2,20210317000664,Y,00131054,유진증권,소액주주,"36,534","36,550",99.96%,"52,922,408","93,839,145",56.40%,2020-12-31,2020,minority_shareholder


In [9]:
# ============================================================
# Cell 9. Process auditor variables
# ============================================================

aud = endpoint_frames.get("auditor_opinion", pd.DataFrame()).copy()

if not aud.empty:
    aud["year"] = aud["query_year"].astype(int)
    
    auditor_col = first_existing_col(aud, ["auditor_nm", "adtor", "auditor", "nm", "adt_instt_nm"])
    opinion_col = first_existing_col(aud, ["adt_opinion", "audit_opinion", "opinion"])
    
    auditor_vars = aud[["corp_code", "year"]].drop_duplicates().copy()
    
    if auditor_col:
        tmp = aud[["corp_code", "year", auditor_col]].dropna().drop_duplicates(["corp_code", "year"])
        tmp = tmp.rename(columns={auditor_col: "AuditorName"})
        auditor_vars = auditor_vars.merge(tmp, on=["corp_code", "year"], how="left")
    else:
        auditor_vars["AuditorName"] = np.nan
    
    if opinion_col:
        tmp = aud[["corp_code", "year", opinion_col]].dropna().drop_duplicates(["corp_code", "year"])
        tmp = tmp.rename(columns={opinion_col: "AuditOpinion"})
        auditor_vars = auditor_vars.merge(tmp, on=["corp_code", "year"], how="left")
    else:
        auditor_vars["AuditOpinion"] = np.nan
    
    big4_keywords = ["삼일", "삼정", "안진", "한영", "pwc", "kpmg", "deloitte", "ey"]
    auditor_vars["Big4Auditor"] = auditor_vars["AuditorName"].astype(str).apply(
        lambda x: int(any(k in x.lower() for k in big4_keywords))
    )
    auditor_vars["CleanOpinion"] = auditor_vars["AuditOpinion"].astype(str).str.contains("적정", na=False).astype(int)
else:
    auditor_vars = pd.DataFrame(columns=["corp_code", "year", "AuditorName", "AuditOpinion", "Big4Auditor", "CleanOpinion"])

print_basic(auditor_vars, "auditor_vars")


auditor_vars
shape: (4631, 6)
columns: ['corp_code', 'year', 'AuditorName', 'AuditOpinion', 'Big4Auditor', 'CleanOpinion']


,corp_code,year,AuditorName,AuditOpinion,Big4Auditor,CleanOpinion
0,00131054,2018,삼정회계법인\n대표이사 김 교 태,연결 : 적정\n별도 : 적정,1,1
1,00131054,2019,삼정회계법인\n대표이사 김 교 태,연결 : 적정\n별도 : 적정,1,0
2,00131054,2020,삼일회계법인\n대표이사 윤 훈 수,연결 : 적정\n별도 : 적정,1,1
3,00131054,2021,삼일회계법인\n대표이사 윤 훈 수,연결 : 적정\n별도 : 적정,1,1
4,00131054,2022,삼일회계법인\n대표이사 윤 훈 수,연결 : 적정\n별도 : 적정,1,1


In [10]:
# ============================================================
# Cell 10. Process audit fee and non-audit fee variables
# ============================================================

audit = endpoint_frames.get("audit_contract", pd.DataFrame()).copy()

if not audit.empty:
    audit["year"] = audit["query_year"].astype(int)
    
    fee_col = first_existing_col(audit, [
        "adt_servc_cncls_mendng",
        "servc_mendng",
        "mendng",
        "cntrct_mendng",
        "adt_cntrct_dtls_mendng"
    ])
    
    hour_col = first_existing_col(audit, [
        "adt_servc_cncls_time",
        "servc_time",
        "tot_reqre_time",
        "adt_cntrct_dtls_time"
    ])
    
    audit["AuditFee"] = audit[fee_col].apply(parse_number) if fee_col else np.nan
    audit["AuditHours"] = audit[hour_col].apply(parse_number) if hour_col else np.nan
    
    audit.loc[audit["AuditFee"] < 0, "AuditFee"] = np.nan
    audit.loc[audit["AuditFee"] > 1e12, "AuditFee"] = np.nan
    
    audit_fee_vars = (
        audit
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            AuditFee=("AuditFee", "sum"),
            AuditHours=("AuditHours", "sum")
        )
    )
    
    audit_fee_vars.loc[audit_fee_vars["AuditFee"] <= 0, "AuditFee"] = np.nan
    audit_fee_vars.loc[audit_fee_vars["AuditHours"] <= 0, "AuditHours"] = np.nan
else:
    audit_fee_vars = pd.DataFrame(columns=["corp_code", "year", "AuditFee", "AuditHours"])


non = endpoint_frames.get("non_audit_contract", pd.DataFrame()).copy()

if not non.empty:
    non["year"] = non["query_year"].astype(int)
    
    non_fee_col = first_existing_col(non, [
        "servc_mendng",
        "mendng",
        "cntrct_mendng",
        "service_fee"
    ])
    
    non["NonAuditFee"] = non[non_fee_col].apply(parse_number) if non_fee_col else np.nan
    
    non.loc[non["NonAuditFee"] < 0, "NonAuditFee"] = np.nan
    non.loc[non["NonAuditFee"] > 1e12, "NonAuditFee"] = np.nan
    
    non_audit_vars = (
        non
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            NonAuditContractCount=("corp_code", "size"),
            NonAuditFee=("NonAuditFee", "sum")
        )
    )
    
    non_audit_vars.loc[non_audit_vars["NonAuditFee"] <= 0, "NonAuditFee"] = 0
else:
    non_audit_vars = pd.DataFrame(columns=["corp_code", "year", "NonAuditContractCount", "NonAuditFee"])

print_basic(audit_fee_vars, "audit_fee_vars")
print_basic(non_audit_vars, "non_audit_vars")


audit_fee_vars
shape: (4632, 4)
columns: ['corp_code', 'year', 'AuditFee', 'AuditHours']


,corp_code,year,AuditFee,AuditHours
0,00100939,2018,224000.0,3484.0
1,00100939,2019,292000.0,3988.0
2,00100939,2020,NaN,NaN
3,00100939,2021,NaN,NaN
4,00100939,2022,NaN,NaN



non_audit_vars
shape: (4632, 4)
columns: ['corp_code', 'year', 'NonAuditContractCount', 'NonAuditFee']


,corp_code,year,NonAuditContractCount,NonAuditFee
0,00100939,2018,3,9000.0
1,00100939,2019,3,10600.0
2,00100939,2020,3,12100.0
3,00100939,2021,3,13600.0
4,00100939,2022,3,13500.0


In [11]:
# ============================================================
# Cell 11. Process governance and ownership variables
# ============================================================

# ------------------------------------------------------------
# Outside directors
# ------------------------------------------------------------

outside = endpoint_frames.get("outside_director", pd.DataFrame()).copy()

if not outside.empty:
    outside["year"] = outside["query_year"].astype(int)
    
    board_col = first_existing_col(outside, ["drctr_co", "director_co", "tot_drctr_co"])
    outside_col = first_existing_col(outside, ["otcmp_drctr_co", "outside_director_co", "outcmpny_drctr_co"])
    
    outside["BoardSize_raw"] = outside[board_col].apply(parse_number) if board_col else np.nan
    outside["OutsideDirectorCount_raw"] = outside[outside_col].apply(parse_number) if outside_col else np.nan
    
    tmp = outside[["corp_code", "year", "BoardSize_raw", "OutsideDirectorCount_raw"]].copy()
    tmp = tmp.dropna(subset=["BoardSize_raw", "OutsideDirectorCount_raw"], how="all")
    
    outside_vars = (
        tmp
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            BoardSize=("BoardSize_raw", "max"),
            OutsideDirectorCount=("OutsideDirectorCount_raw", "max")
        )
    )
    
    outside_vars["OutsideDirectorRatio"] = outside_vars.apply(
        lambda r: safe_divide(r["OutsideDirectorCount"], r["BoardSize"]),
        axis=1
    )
    
    outside_vars.loc[outside_vars["BoardSize"] <= 0, "BoardSize"] = np.nan
    outside_vars.loc[outside_vars["OutsideDirectorCount"] < 0, "OutsideDirectorCount"] = np.nan
    outside_vars.loc[outside_vars["OutsideDirectorRatio"] < 0, "OutsideDirectorRatio"] = np.nan
    outside_vars.loc[outside_vars["OutsideDirectorRatio"] > 1, "OutsideDirectorRatio"] = np.nan
else:
    outside_vars = pd.DataFrame(columns=["corp_code", "year", "BoardSize", "OutsideDirectorCount", "OutsideDirectorRatio"])


# ------------------------------------------------------------
# Largest shareholder
# ------------------------------------------------------------

largest = endpoint_frames.get("largest_shareholder", pd.DataFrame()).copy()

if not largest.empty:
    largest["year"] = largest["query_year"].astype(int)
    
    own_col = first_existing_col(largest, [
        "trmend_posesn_stock_qota_rt",
        "posesn_stock_qota_rt",
        "qota_rt",
        "stock_qota_rt",
        "rt"
    ])
    
    largest["LargestShareholderOwnership"] = largest[own_col].apply(parse_number) if own_col else np.nan
    
    largest_vars = (
        largest
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            LargestShareholderOwnership=("LargestShareholderOwnership", "max")
        )
    )
    
    largest_vars.loc[largest_vars["LargestShareholderOwnership"] < 0, "LargestShareholderOwnership"] = np.nan
    largest_vars.loc[largest_vars["LargestShareholderOwnership"] > 100, "LargestShareholderOwnership"] = np.nan
else:
    largest_vars = pd.DataFrame(columns=["corp_code", "year", "LargestShareholderOwnership"])


# ------------------------------------------------------------
# Minority shareholders
# ------------------------------------------------------------

minority = endpoint_frames.get("minority_shareholder", pd.DataFrame()).copy()

if not minority.empty:
    minority["year"] = minority["query_year"].astype(int)
    
    ratio_col = first_existing_col(minority, [
        "mrhl_rate",
        "qota_rt",
        "rt"
    ])
    
    count_col = first_existing_col(minority, [
        "shrholdr_co",
        "mrhl_co",
        "stockholdr_co"
    ])
    
    minority["MinorityShareholderRatio"] = minority[ratio_col].apply(parse_number) if ratio_col else np.nan
    minority["MinorityShareholderCount"] = minority[count_col].apply(parse_number) if count_col else np.nan
    
    minority_vars = (
        minority
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            MinorityShareholderRatio=("MinorityShareholderRatio", "max"),
            MinorityShareholderCount=("MinorityShareholderCount", "max")
        )
    )
    
    minority_vars.loc[minority_vars["MinorityShareholderRatio"] < 0, "MinorityShareholderRatio"] = np.nan
    minority_vars.loc[minority_vars["MinorityShareholderRatio"] > 100, "MinorityShareholderRatio"] = np.nan
else:
    minority_vars = pd.DataFrame(columns=["corp_code", "year", "MinorityShareholderRatio", "MinorityShareholderCount"])

print_basic(outside_vars, "outside_vars")
print_basic(largest_vars, "largest_vars")
print_basic(minority_vars, "minority_vars")


outside_vars
shape: (3343, 5)
columns: ['corp_code', 'year', 'BoardSize', 'OutsideDirectorCount', 'OutsideDirectorRatio']


,corp_code,year,BoardSize,OutsideDirectorCount,OutsideDirectorRatio
0,00100939,2020,6.0,3.0,0.500000
1,00100939,2021,6.0,3.0,0.500000
2,00100939,2022,6.0,3.0,0.500000
3,00100939,2023,6.0,3.0,0.500000
4,00100939,2024,7.0,3.0,0.428571



largest_vars
shape: (4632, 3)
columns: ['corp_code', 'year', 'LargestShareholderOwnership']


,corp_code,year,LargestShareholderOwnership
0,00100939,2018,50.74
1,00100939,2019,50.74
2,00100939,2020,50.74
3,00100939,2021,50.74
4,00100939,2022,50.74



minority_vars
shape: (4632, 4)
columns: ['corp_code', 'year', 'MinorityShareholderRatio', 'MinorityShareholderCount']


,corp_code,year,MinorityShareholderRatio,MinorityShareholderCount
0,00100939,2018,NaN,2678.0
1,00100939,2019,NaN,3013.0
2,00100939,2020,NaN,3272.0
3,00100939,2021,NaN,5415.0
4,00100939,2022,NaN,6413.0


In [12]:
# ============================================================
# Cell 12. Collect financial statement key accounts
# ============================================================

def fetch_financial_accounts(corp_code, year):
    params = {
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": REPORT_CODE
    }
    return dart_list("fnlttSinglAcnt.json", params)

fin_rows = []

for _, firm in tqdm(sample_dart.iterrows(), total=len(sample_dart), desc="financials"):
    for year in YEARS:
        out = fetch_financial_accounts(firm["corp_code"], year)
        for item in out:
            item["query_year"] = year
            fin_rows.append(item)

fin_raw = pd.DataFrame(fin_rows)

print_basic(fin_raw, "fin_raw")
if not fin_raw.empty:
    print(fin_raw["account_nm"].value_counts().head(50))

save_df(fin_raw, "raw_financial_accounts.csv")

financials:   0%|          | 0/711 [00:00<?, ?it/s]


fin_raw
shape: (121713, 22)
columns: ['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'stock_code', 'fs_div', 'fs_nm', 'sj_div', 'sj_nm', 'account_nm', 'thstrm_nm', 'thstrm_dt', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_dt', 'frmtrm_amount', 'bfefrmtrm_nm', 'bfefrmtrm_dt', 'bfefrmtrm_amount', 'ord', 'currency', 'query_year']


,rcept_no,reprt_code,bsns_year,corp_code,stock_code,fs_div,fs_nm,sj_div,sj_nm,account_nm,...,thstrm_amount,frmtrm_nm,frmtrm_dt,frmtrm_amount,bfefrmtrm_nm,bfefrmtrm_dt,bfefrmtrm_amount,ord,currency,query_year
0,20240318000590,11011,2023,00131054,001200,CFS,연결재무제표,BS,재무상태표,자산총계,...,"8,699,332,043,240",제 70 기,2022.12.31 현재,"8,076,408,298,781",제 69 기,2021.12.31 현재,"9,175,464,956,273",5,KRW,2023
1,20240318000590,11011,2023,00131054,001200,CFS,연결재무제표,BS,재무상태표,부채총계,...,"7,687,427,343,652",제 70 기,2022.12.31 현재,"7,100,764,279,436",제 69 기,2021.12.31 현재,"8,214,653,815,670",11,KRW,2023
2,20240318000590,11011,2023,00131054,001200,CFS,연결재무제표,BS,재무상태표,자본금,...,"537,592,090,000",제 70 기,2022.12.31 현재,"537,592,090,000",제 69 기,2021.12.31 현재,"537,592,090,000",13,KRW,2023
3,20240318000590,11011,2023,00131054,001200,CFS,연결재무제표,BS,재무상태표,이익잉여금,...,"343,979,785,650",제 70 기,2022.12.31 현재,"318,793,659,102",제 69 기,2021.12.31 현재,"313,097,862,676",17,KRW,2023
4,20240318000590,11011,2023,00131054,001200,CFS,연결재무제표,BS,재무상태표,자본총계,...,"1,011,904,699,588",제 70 기,2022.12.31 현재,"975,644,019,345",제 69 기,2021.12.31 현재,"960,811,140,603",21,KRW,2023


account_nm
당기순이익(손실)            16944
부채총계                  8534
자산총계                  8531
유동자산                  8530
비유동자산                 8530
유동부채                  8530
비유동부채                 8530
영업이익                  8515
법인세차감전 순이익            8503
자본총계                  8474
매출액                   8431
이익잉여금                 8396
자본금                   8325
총포괄손익                 2587
영업비용                    97
기타포괄손익-공정가치측정금융자산       58
당기손익-공정가치측정금융자산         54
차입부채                    32
상각후원가측정금융자산             26
파생상품자산                  25
파생상품부채                  20
이자수익                    15
이자비용                     7
금융상품관련순손익                6
예수부채                     5
영업이익(손실)                 5
금융부채                     3
Name: count, dtype: int64
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\raw_financial_accounts.csv | shape=(121713, 22)


WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/raw_financial_accounts.csv')

In [13]:
# ============================================================
# Cell 13. Process financial variables
# ============================================================

def select_account_amount(df, patterns, prefer_sj=None):
    if df.empty:
        return pd.DataFrame(columns=["corp_code", "year", "value"])
    
    d = df.copy()
    d["account_nm"] = d["account_nm"].astype(str)
    
    mask = False
    for p in patterns:
        mask = mask | d["account_nm"].str.contains(p, na=False, regex=False)
    
    d = d[mask].copy()
    
    if prefer_sj and "sj_div" in d.columns:
        d = d[d["sj_div"].eq(prefer_sj)].copy()
    
    if d.empty:
        return pd.DataFrame(columns=["corp_code", "year", "value"])
    
    d["year"] = d["query_year"].astype(int)
    d["amount"] = d["thstrm_amount"].apply(parse_number)
    
    if "fs_div" in d.columns:
        d["fs_priority"] = np.where(d["fs_div"].eq("CFS"), 2, 1)
    else:
        d["fs_priority"] = 1
    
    def account_priority(name):
        for i, p in enumerate(patterns):
            if p in str(name):
                return i
        return 999
    
    d["account_priority"] = d["account_nm"].apply(account_priority)
    d["nonmissing_amount"] = d["amount"].notna().astype(int)
    
    d = d.sort_values(
        ["corp_code", "year", "nonmissing_amount", "fs_priority", "account_priority"],
        ascending=[True, True, False, False, True]
    )
    
    out = d.groupby(["corp_code", "year"], as_index=False)["amount"].first()
    out = out.rename(columns={"amount": "value"})
    return out


def make_fin_vars(fin_raw):
    if fin_raw.empty:
        return pd.DataFrame(columns=[
            "corp_code", "year", "Assets", "Liabilities", "Equity",
            "Revenue", "OperatingIncome", "NetIncome"
        ])
    
    df = fin_raw.copy()
    df["query_year"] = df["query_year"].astype(int)
    
    base = df[["corp_code", "query_year"]].drop_duplicates()
    base = base.rename(columns={"query_year": "year"})
    
    specs = {
        "Assets": {
            "patterns": ["자산총계"],
            "sj": "BS"
        },
        "Liabilities": {
            "patterns": ["부채총계"],
            "sj": "BS"
        },
        "Equity": {
            "patterns": ["자본총계"],
            "sj": "BS"
        },
        "Revenue": {
            "patterns": ["매출액", "영업수익", "수익(매출액)", "매출"],
            "sj": "IS"
        },
        "OperatingIncome": {
            "patterns": ["영업이익"],
            "sj": "IS"
        },
        "NetIncome": {
            "patterns": [
                "당기순이익",
                "당기순이익(손실)",
                "연결당기순이익",
                "지배기업의 소유주에게 귀속되는 당기순이익",
                "지배기업소유주지분순이익",
                "순이익"
            ],
            "sj": "IS"
        }
    }
    
    out = base.copy()
    
    for var, spec in specs.items():
        tmp = select_account_amount(df, spec["patterns"], prefer_sj=spec["sj"])
        tmp = tmp.rename(columns={"value": var})
        out = out.merge(tmp, on=["corp_code", "year"], how="left")
    
    out["LogAssets"] = np.log(out["Assets"].replace(0, np.nan))
    out["Leverage"] = out.apply(lambda r: safe_divide(r["Liabilities"], r["Assets"]), axis=1)
    out["ROA"] = out.apply(lambda r: safe_divide(r["NetIncome"], r["Assets"]), axis=1)
    out["Loss"] = np.where(out["NetIncome"].notna(), (out["NetIncome"] < 0).astype(int), np.nan)
    
    out.loc[out["Assets"] <= 0, ["Assets", "LogAssets"]] = np.nan
    out.loc[(out["Leverage"] < 0) | (out["Leverage"] > 2), "Leverage"] = np.nan
    out.loc[(out["ROA"] < -2) | (out["ROA"] > 2), "ROA"] = np.nan
    
    return out


fin_vars = make_fin_vars(fin_raw)

print_basic(fin_vars, "fin_vars")
print(fin_vars[["Assets", "Revenue", "NetIncome", "ROA"]].describe())

save_df(fin_vars, "financial_vars.csv")


fin_vars
shape: (4611, 12)
columns: ['corp_code', 'year', 'Assets', 'Liabilities', 'Equity', 'Revenue', 'OperatingIncome', 'NetIncome', 'LogAssets', 'Leverage', 'ROA', 'Loss']


,corp_code,year,Assets,Liabilities,Equity,Revenue,OperatingIncome,NetIncome,LogAssets,Leverage,ROA,Loss
0,00131054,2023,8.699332e+12,7.687427e+12,1.011905e+12,1.647663e+12,2.703937e+10,3.069648e+10,29.794267,0.883680,0.003529,0.0
1,00131054,2024,9.646708e+12,8.595332e+12,1.051376e+12,1.652600e+12,5.833814e+10,4.961933e+10,29.897638,0.891012,0.005144,0.0
2,01649204,2024,1.361082e+11,3.403909e+10,1.020691e+11,3.090626e+11,5.984471e+10,1.540898e+10,25.636716,0.250089,0.113211,0.0
3,01210677,2024,1.884231e+11,9.729188e+10,9.113119e+10,2.957725e+11,1.652431e+10,1.981255e+10,25.961956,0.516348,0.105149,0.0
4,00139834,2023,4.040680e+12,2.172423e+12,1.868256e+12,5.605300e+12,4.640484e+11,3.323522e+11,29.027434,0.537638,0.082252,0.0


             Assets       Revenue     NetIncome          ROA
count  4.611000e+03  4.579000e+03  4.611000e+03  4610.000000
mean   4.800658e+12  3.256292e+12  1.488433e+11     0.014675
std    2.391365e+13  1.408416e+13  1.637405e+12     0.091053
min    1.265509e+08  0.000000e+00 -2.442911e+13    -1.375255
25%    2.498956e+11  1.840731e+11 -9.489853e+08    -0.003093
50%    5.776104e+11  4.941579e+11  1.176710e+10     0.023248
75%    1.764403e+12  1.555805e+12  5.188627e+10     0.051777
max    5.145319e+14  3.022314e+14  5.565408e+13     1.718552
Saved: c:\Users\starw\.vscode\practice\output_governance_direction\financial_vars.csv | shape=(4611, 12)


WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/financial_vars.csv')

In [14]:
# ============================================================
# Cell 14. Merge final panel
# ============================================================

final_panel = panel.copy()

for df in [
    auditor_vars,
    audit_fee_vars,
    non_audit_vars,
    outside_vars,
    largest_vars,
    minority_vars,
    fin_vars
]:
    if df is not None and not df.empty:
        final_panel = final_panel.merge(df, on=["corp_code", "year"], how="left")

# 비감사용역은 없으면 0으로 처리
final_panel["NonAuditFee"] = final_panel["NonAuditFee"].fillna(0)
final_panel["NonAuditContractCount"] = final_panel["NonAuditContractCount"].fillna(0)

final_panel["NonAuditFeeRatio"] = final_panel.apply(
    lambda r: safe_divide(r["NonAuditFee"], r["AuditFee"]),
    axis=1
)

final_panel.loc[final_panel["NonAuditFeeRatio"] < 0, "NonAuditFeeRatio"] = np.nan
final_panel.loc[final_panel["NonAuditFeeRatio"] > 5, "NonAuditFeeRatio"] = np.nan

final_panel["LogAuditFee"] = np.log(final_panel["AuditFee"].replace(0, np.nan))
final_panel["LogNonAuditFee"] = np.log1p(final_panel["NonAuditFee"])

# winsorize
for c in [
    "NonAuditFeeRatio", "LogAuditFee", "LogNonAuditFee",
    "OutsideDirectorRatio", "LargestShareholderOwnership", "MinorityShareholderRatio",
    "MinorityShareholderCount", "LogAssets", "Leverage", "ROA"
]:
    if c in final_panel.columns:
        final_panel[c + "_w"] = winsorize_series(final_panel[c])

print_basic(final_panel, "final_panel", n=10)

key_check = [
    "CorrectionDummy", "CorrectionCount", "MaterialCorrectionDummy",
    "AuditFee", "NonAuditFee", "NonAuditFeeRatio",
    "OutsideDirectorRatio", "LargestShareholderOwnership", "MinorityShareholderRatio",
    "MinorityShareholderCount", "LogAssets", "Leverage", "ROA"
]

key_check = [c for c in key_check if c in final_panel.columns]

print(final_panel[key_check].describe().T)
print("\nMissing rates:")
print(final_panel[key_check].isna().mean().sort_values(ascending=False))

save_df(final_panel, "final_dart_correction_panel_governance_direction.csv")


final_panel
shape: (4977, 51)
columns: ['stock_code', 'corp_code', 'corp_name', 'market', 'sector', 'listing_date', 'year', 'CorrectionCount', 'FirstCorrectionDate', 'LastCorrectionDate', 'CorrectionDummy', 'MaterialCorrectionCount', 'MaterialKeywordHits', 'MaterialCorrectionDummy', 'AuditorName', 'AuditOpinion', 'Big4Auditor', 'CleanOpinion', 'AuditFee', 'AuditHours', 'NonAuditContractCount', 'NonAuditFee', 'BoardSize', 'OutsideDirectorCount', 'OutsideDirectorRatio', 'LargestShareholderOwnership', 'MinorityShareholderRatio', 'MinorityShareholderCount', 'Assets', 'Liabilities', 'Equity', 'Revenue', 'OperatingIncome', 'NetIncome', 'LogAssets', 'Leverage', 'ROA', 'Loss', 'NonAuditFeeRatio', 'LogAuditFee', 'LogNonAuditFee', 'NonAuditFeeRatio_w', 'LogAuditFee_w', 'LogNonAuditFee_w', 'OutsideDirectorRatio_w', 'LargestShareholderOwnership_w', 'MinorityShareholderRatio_w', 'MinorityShareholderCount_w', 'LogAssets_w', 'Leverage_w', 'ROA_w']


,stock_code,corp_code,corp_name,market,sector,listing_date,year,CorrectionCount,FirstCorrectionDate,LastCorrectionDate,...,NonAuditFeeRatio_w,LogAuditFee_w,LogNonAuditFee_w,OutsideDirectorRatio_w,LargestShareholderOwnership_w,MinorityShareholderRatio_w,MinorityShareholderCount_w,LogAssets_w,Leverage_w,ROA_w
0,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2018,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2019,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2020,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2021,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2022,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2023,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2024,0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,001200,00131054,삼양바이오팜,KOSPI,의약품 제조업,2025-11-24,2018,0,NaN,NaN,...,0.134940,6.028279,4.043051,NaN,28.70,NaN,38356.0,NaN,NaN,NaN
8,001200,00131054,삼양바이오팜,KOSPI,의약품 제조업,2025-11-24,2019,0,NaN,NaN,...,0.116641,6.466145,4.330733,NaN,28.80,NaN,36199.0,NaN,NaN,NaN
9,001200,00131054,삼양바이오팜,KOSPI,의약품 제조업,2025-11-24,2020,0,NaN,NaN,...,NaN,NaN,3.583519,0.6,29.03,NaN,36534.0,NaN,NaN,NaN


                              count          mean           std        min  \
CorrectionDummy              4977.0  1.993169e-01  3.995269e-01   0.000000   
CorrectionCount              4977.0  2.943540e-01  8.265796e-01   0.000000   
MaterialCorrectionDummy      4977.0  6.951979e-02  2.543615e-01   0.000000   
AuditFee                     1188.0  3.977972e+07  1.252495e+08   3.000000   
NonAuditFee                  4977.0  3.481641e+06  3.065002e+07   0.000000   
NonAuditFeeRatio             1155.0  1.038003e-01  3.219452e-01   0.000000   
OutsideDirectorRatio         3322.0  4.390439e-01  1.495792e-01   0.000000   
LargestShareholderOwnership  4632.0  4.442078e+01  1.658194e+01   2.000000   
MinorityShareholderRatio        0.0           NaN           NaN        NaN   
MinorityShareholderCount     4449.0  4.658150e+04  1.886832e+05  10.000000   
LogAssets                    4611.0  2.732183e+01  1.661513e+00  18.656155   
Leverage                     4610.0  4.621888e-01  2.092886e-01 

WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/final_dart_correction_panel_governance_direction.csv')

In [15]:
# ============================================================
# Cell 15. Descriptive statistics
# ============================================================

desc_vars = [
    "CorrectionDummy", "CorrectionCount", "MaterialCorrectionDummy", "MaterialCorrectionCount",
    "OutsideDirectorRatio", "BoardSize", "OutsideDirectorCount",
    "LargestShareholderOwnership", "MinorityShareholderRatio", "MinorityShareholderCount",
    "LogAssets", "Leverage", "ROA", "Loss",
    "Big4Auditor", "CleanOpinion", "NonAuditFeeRatio", "LogAuditFee", "LogNonAuditFee"
]

desc_vars = [c for c in desc_vars if c in final_panel.columns]

desc = final_panel[desc_vars].describe().T
display(desc)
save_df(desc.reset_index().rename(columns={"index": "variable"}), "table_01_descriptive_statistics.csv")

yearly = (
    final_panel
    .groupby("year", as_index=False)
    .agg(
        n_firm_years=("corp_code", "count"),
        n_firms=("corp_code", "nunique"),
        correction_rate=("CorrectionDummy", "mean"),
        avg_correction_count=("CorrectionCount", "mean"),
        material_rate=("MaterialCorrectionDummy", "mean")
    )
)

display(yearly)
save_df(yearly, "table_02_yearly_correction.csv")

missing = final_panel[desc_vars].isna().mean().sort_values(ascending=False).to_frame("missing_rate")
display(missing)
save_df(missing.reset_index().rename(columns={"index": "variable"}), "table_03_missing_rates.csv")

,count,mean,std,min,25%,50%,75%,max
CorrectionDummy,4977.0,0.199317,0.399527,0.000000,0.000000,0.000000,0.000000,1.000000e+00
CorrectionCount,4977.0,0.294354,0.826580,0.000000,0.000000,0.000000,0.000000,1.300000e+01
MaterialCorrectionDummy,4977.0,0.069520,0.254362,0.000000,0.000000,0.000000,0.000000,1.000000e+00
MaterialCorrectionCount,4977.0,0.093631,0.454605,0.000000,0.000000,0.000000,0.000000,1.000000e+01
OutsideDirectorRatio,3322.0,0.439044,0.149579,0.000000,0.333333,0.428571,0.571429,1.000000e+00
BoardSize,3340.0,5.784431,2.008874,1.000000,4.000000,6.000000,7.000000,1.500000e+01
OutsideDirectorCount,3334.0,2.630774,1.411292,0.000000,1.000000,3.000000,3.000000,9.000000e+00
LargestShareholderOwnership,4632.0,44.420775,16.581939,2.000000,32.777500,45.010000,55.445000,1.000000e+02
MinorityShareholderRatio,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MinorityShareholderCount,4449.0,46581.501011,188683.203460,10.000000,6976.000000,14815.000000,32057.000000,5.813977e+06


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\table_01_descriptive_statistics.csv | shape=(19, 9)


,year,n_firm_years,n_firms,correction_rate,avg_correction_count,material_rate
0,2018,711,711,0.205345,0.350211,0.000000
1,2019,711,711,0.184248,0.270042,0.000000
2,2020,711,711,0.123769,0.187060,0.001406
3,2021,711,711,0.229255,0.353024,0.001406
4,2022,711,711,0.261603,0.354430,0.106892
5,2023,711,711,0.165963,0.230661,0.161744
6,2024,711,711,0.225035,0.315049,0.215190


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\table_02_yearly_correction.csv | shape=(7, 6)


,missing_rate
MinorityShareholderRatio,1.000000
NonAuditFeeRatio,0.767932
LogAuditFee,0.761302
OutsideDirectorRatio,0.332530
OutsideDirectorCount,0.330119
BoardSize,0.328913
MinorityShareholderCount,0.106088
ROA,0.073739
Leverage,0.073739
LogAssets,0.073538


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\table_03_missing_rates.csv | shape=(19, 2)


WindowsPath('c:/Users/starw/.vscode/practice/output_governance_direction/table_03_missing_rates.csv')

In [24]:
# ============================================================
# Revised Cell A. Prepare regression variables correctly
# ============================================================

reg = final_panel.copy()
reg = reg.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# 1. 소액주주 수 로그변환
# ------------------------------------------------------------

if "MinorityShareholderCount" in reg.columns:
    reg["LogMinorityShareholderCount"] = np.log1p(reg["MinorityShareholderCount"])
    reg["LogMinorityShareholderCount_w"] = winsorize_series(reg["LogMinorityShareholderCount"])

# ------------------------------------------------------------
# 2. 기본 통제변수
# ------------------------------------------------------------

base_controls = [
    "LogAssets_w",
    "Leverage_w",
    "ROA_w",
    "Loss"
]
base_controls = [c for c in base_controls if c in reg.columns]

# ------------------------------------------------------------
# 3. 지배구조·소유구조 변수
# MinorityShareholderRatio는 전부 결측이므로 제외
# ------------------------------------------------------------

governance_vars = []

if "OutsideDirectorRatio_w" in reg.columns and reg["OutsideDirectorRatio_w"].notna().sum() > 100:
    governance_vars.append("OutsideDirectorRatio_w")

if "LargestShareholderOwnership_w" in reg.columns and reg["LargestShareholderOwnership_w"].notna().sum() > 100:
    governance_vars.append("LargestShareholderOwnership_w")

if "LogMinorityShareholderCount_w" in reg.columns and reg["LogMinorityShareholderCount_w"].notna().sum() > 100:
    governance_vars.append("LogMinorityShareholderCount_w")

# ------------------------------------------------------------
# 4. 감사인 보조변수
# 결측이 심한 NonAuditFeeRatio는 별도 보조모형에서만 사용
# ------------------------------------------------------------

auditor_basic_vars = []

if "Big4Auditor" in reg.columns and reg["Big4Auditor"].notna().sum() > 100:
    auditor_basic_vars.append("Big4Auditor")

if "CleanOpinion" in reg.columns and reg["CleanOpinion"].notna().sum() > 100:
    auditor_basic_vars.append("CleanOpinion")

audit_fee_vars = []

if "NonAuditFeeRatio_w" in reg.columns and reg["NonAuditFeeRatio_w"].notna().sum() > 100:
    audit_fee_vars.append("NonAuditFeeRatio_w")

if "LogAuditFee_w" in reg.columns and reg["LogAuditFee_w"].notna().sum() > 100:
    audit_fee_vars.append("LogAuditFee_w")

if "LogNonAuditFee_w" in reg.columns and reg["LogNonAuditFee_w"].notna().sum() > 100:
    audit_fee_vars.append("LogNonAuditFee_w")

print("base_controls:", base_controls)
print("governance_vars:", governance_vars)
print("auditor_basic_vars:", auditor_basic_vars)
print("audit_fee_vars:", audit_fee_vars)

print("\nNon-missing counts:")
for v in base_controls + governance_vars + auditor_basic_vars + audit_fee_vars:
    print(v, reg[v].notna().sum())

base_controls: ['LogAssets_w', 'Leverage_w', 'ROA_w', 'Loss']
governance_vars: ['OutsideDirectorRatio_w', 'LargestShareholderOwnership_w', 'LogMinorityShareholderCount_w']
auditor_basic_vars: ['Big4Auditor', 'CleanOpinion']
audit_fee_vars: ['NonAuditFeeRatio_w', 'LogAuditFee_w', 'LogNonAuditFee_w']

Non-missing counts:
LogAssets_w 4611
Leverage_w 4610
ROA_w 4610
Loss 4611
OutsideDirectorRatio_w 3322
LargestShareholderOwnership_w 4632
LogMinorityShareholderCount_w 4449
Big4Auditor 4631
CleanOpinion 4631
NonAuditFeeRatio_w 1155
LogAuditFee_w 1188
LogNonAuditFee_w 4977


In [25]:
# ============================================================
# Revised Cell B. Regression functions
# ============================================================

regression_results = {}
regression_samples = {}

def run_lpm_revised(dep, xvars, name, add_fe=True):
    use_cols = [dep, "corp_code", "year", "sector"] + xvars
    use_cols = [c for c in use_cols if c in reg.columns]
    
    df = reg[use_cols].dropna().copy()
    
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("N:", len(df))
    print("Firms:", df["corp_code"].nunique() if "corp_code" in df.columns else None)
    print("Dep mean:", df[dep].mean() if dep in df.columns and len(df) > 0 else None)
    
    if len(df) < 50 or df[dep].nunique() < 2:
        print("[SKIPPED] 데이터 부족 또는 종속변수 variation 부족")
        return None, df
    
    rhs_vars = xvars.copy()
    if add_fe:
        rhs_vars = rhs_vars + ["C(year)", "C(sector)"]
    
    formula = f"{dep} ~ {' + '.join(rhs_vars)}"
    
    model = smf.ols(formula, data=df).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["corp_code"]}
    )
    
    print(model.summary())
    return model, df


def run_poisson_revised(dep, xvars, name, add_fe=True):
    use_cols = [dep, "corp_code", "year", "sector"] + xvars
    use_cols = [c for c in use_cols if c in reg.columns]
    
    df = reg[use_cols].dropna().copy()
    
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("N:", len(df))
    print("Firms:", df["corp_code"].nunique() if "corp_code" in df.columns else None)
    print("Dep mean:", df[dep].mean() if dep in df.columns and len(df) > 0 else None)
    
    if len(df) < 50 or df[dep].sum() <= 0:
        print("[SKIPPED] 데이터 부족 또는 count variation 부족")
        return None, df
    
    rhs_vars = xvars.copy()
    if add_fe:
        rhs_vars = rhs_vars + ["C(year)", "C(sector)"]
    
    formula = f"{dep} ~ {' + '.join(rhs_vars)}"
    
    try:
        model = smf.poisson(formula, data=df).fit(maxiter=300, disp=False)
        print(model.summary())
        return model, df
    except Exception as e:
        print("[FAILED]", e)
        return None, df

In [26]:
# ============================================================
# Revised Cell C. Main regressions
# ============================================================

# ------------------------------------------------------------
# CorrectionDummy
# ------------------------------------------------------------

m1, df_m1 = run_lpm_revised(
    dep="CorrectionDummy",
    xvars=base_controls,
    name="M1 Controls only: CorrectionDummy"
)
regression_results["M1_controls"] = m1
regression_samples["M1_controls"] = df_m1

m2, df_m2 = run_lpm_revised(
    dep="CorrectionDummy",
    xvars=base_controls + governance_vars,
    name="M2 Governance and ownership: CorrectionDummy"
)
regression_results["M2_governance"] = m2
regression_samples["M2_governance"] = df_m2

m3, df_m3 = run_lpm_revised(
    dep="CorrectionDummy",
    xvars=base_controls + governance_vars + auditor_basic_vars,
    name="M3 Governance + auditor basic: CorrectionDummy"
)
regression_results["M3_governance_auditor_basic"] = m3
regression_samples["M3_governance_auditor_basic"] = df_m3

m4, df_m4 = run_lpm_revised(
    dep="CorrectionDummy",
    xvars=base_controls + governance_vars + auditor_basic_vars + audit_fee_vars,
    name="M4 Governance + auditor + audit fees: CorrectionDummy"
)
regression_results["M4_audit_fee_aux"] = m4
regression_samples["M4_audit_fee_aux"] = df_m4


# ------------------------------------------------------------
# MaterialCorrectionDummy
# ------------------------------------------------------------

m5, df_m5 = run_lpm_revised(
    dep="MaterialCorrectionDummy",
    xvars=base_controls,
    name="M5 Controls only: MaterialCorrectionDummy"
)
regression_results["M5_material_controls"] = m5
regression_samples["M5_material_controls"] = df_m5

m6, df_m6 = run_lpm_revised(
    dep="MaterialCorrectionDummy",
    xvars=base_controls + governance_vars,
    name="M6 Governance and ownership: MaterialCorrectionDummy"
)
regression_results["M6_material_governance"] = m6
regression_samples["M6_material_governance"] = df_m6

m7, df_m7 = run_lpm_revised(
    dep="MaterialCorrectionDummy",
    xvars=base_controls + governance_vars + auditor_basic_vars,
    name="M7 Governance + auditor basic: MaterialCorrectionDummy"
)
regression_results["M7_material_auditor_basic"] = m7
regression_samples["M7_material_auditor_basic"] = df_m7


# ------------------------------------------------------------
# CorrectionCount
# ------------------------------------------------------------

p1, df_p1 = run_poisson_revised(
    dep="CorrectionCount",
    xvars=base_controls,
    name="P1 Controls only: CorrectionCount"
)
regression_results["P1_count_controls"] = p1
regression_samples["P1_count_controls"] = df_p1

p2, df_p2 = run_poisson_revised(
    dep="CorrectionCount",
    xvars=base_controls + governance_vars,
    name="P2 Governance and ownership: CorrectionCount"
)
regression_results["P2_count_governance"] = p2
regression_samples["P2_count_governance"] = df_p2

p3, df_p3 = run_poisson_revised(
    dep="CorrectionCount",
    xvars=base_controls + governance_vars + auditor_basic_vars,
    name="P3 Governance + auditor basic: CorrectionCount"
)
regression_results["P3_count_auditor_basic"] = p3
regression_samples["P3_count_auditor_basic"] = df_p3


M1 Controls only: CorrectionDummy
N: 4610
Firms: 689
Dep mean: 0.21344902386117137
                            OLS Regression Results                            
Dep. Variable:        CorrectionDummy   R-squared:                       0.057
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                 3.212e+04
Date:                Fri, 05 Jun 2026   Prob (F-statistic):               0.00
Time:                        01:02:23   Log-Likelihood:                -2291.7
No. Observations:                4610   AIC:                             4841.
Df Residuals:                    4481   BIC:                             5672.
Df Model:                         128                                         
Covariance Type:              cluster                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------

In [27]:
# ============================================================
# Revised Cell D. Save coefficient tables
# ============================================================

coef_tables = []

for name, model in regression_results.items():
    if model is None:
        continue
    
    tmp = pd.DataFrame({
        "model": name,
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t_or_z": model.tvalues.values,
        "p_value": model.pvalues.values,
        "nobs": model.nobs
    })
    coef_tables.append(tmp)

if coef_tables:
    coef_table = pd.concat(coef_tables, ignore_index=True)
else:
    coef_table = pd.DataFrame(columns=["model", "variable", "coef", "std_err", "t_or_z", "p_value", "nobs"])

display(coef_table)
save_df(coef_table, "revised_table_04_regression_coefficients.csv")

sample_rows = []

for name, df in regression_samples.items():
    dep_col = df.columns[0] if not df.empty else None
    sample_rows.append({
        "model": name,
        "n_obs": len(df),
        "n_firms": df["corp_code"].nunique() if "corp_code" in df.columns and not df.empty else np.nan,
        "dep_mean": df[dep_col].mean() if dep_col and not df.empty else np.nan
    })

reg_sample_table = pd.DataFrame(sample_rows)
display(reg_sample_table)
save_df(reg_sample_table, "revised_table_05_regression_sample_sizes.csv")

for name, model in regression_results.items():
    if model is not None:
        with open(OUT_DIR / f"revised_regression_{name}.txt", "w", encoding="utf-8") as f:
            f.write(model.summary().as_text())

,model,variable,coef,std_err,t_or_z,p_value,nobs
0,M1_controls,Intercept,-0.428887,0.155640,-2.755638,0.005858,4610.0
1,M1_controls,C(year)[T.2019],-0.030051,0.022934,-1.310323,0.190087,4610.0
2,M1_controls,C(year)[T.2020],-0.099590,0.021076,-4.725329,0.000002,4610.0
3,M1_controls,C(year)[T.2021],0.017698,0.022765,0.777416,0.436914,4610.0
4,M1_controls,C(year)[T.2022],0.045144,0.024284,1.858982,0.063030,4610.0
...,...,...,...,...,...,...,...
1174,P3_count_auditor_basic,OutsideDirectorRatio_w,NaN,NaN,NaN,NaN,3217.0
1175,P3_count_auditor_basic,LargestShareholderOwnership_w,NaN,NaN,NaN,NaN,3217.0
1176,P3_count_auditor_basic,LogMinorityShareholderCount_w,NaN,NaN,NaN,NaN,3217.0
1177,P3_count_auditor_basic,Big4Auditor,NaN,NaN,NaN,NaN,3217.0


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\revised_table_04_regression_coefficients.csv | shape=(1179, 7)


,model,n_obs,n_firms,dep_mean
0,M1_controls,4610,689,0.213449
1,M2_governance,3218,682,0.211933
2,M3_governance_auditor_basic,3217,682,0.211999
3,M4_audit_fee_aux,1,1,0.000000
4,M5_material_controls,4610,689,0.074837
5,M6_material_governance,3218,682,0.104723
6,M7_material_auditor_basic,3217,682,0.104756
7,P1_count_controls,4610,689,0.313883
8,P2_count_governance,3218,682,0.305780
9,P3_count_auditor_basic,3217,682,0.305875


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\revised_table_05_regression_sample_sizes.csv | shape=(10, 4)


In [28]:
# ============================================================
# Revised Cell E. Paper-style key coefficient tables
# ============================================================

important_vars = [
    "OutsideDirectorRatio_w",
    "LargestShareholderOwnership_w",
    "LogMinorityShareholderCount_w",
    "LogAssets_w",
    "Leverage_w",
    "ROA_w",
    "Loss",
    "Big4Auditor",
    "CleanOpinion",
    "NonAuditFeeRatio_w",
    "LogAuditFee_w",
    "LogNonAuditFee_w"
]

paper_coef = coef_table[coef_table["variable"].isin(important_vars)].copy()

def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

paper_coef["stars"] = paper_coef["p_value"].apply(stars)
paper_coef["coef_stars"] = paper_coef.apply(lambda r: f"{r['coef']:.4f}{r['stars']}", axis=1)
paper_coef["std_err_fmt"] = paper_coef["std_err"].apply(lambda x: f"({x:.4f})" if pd.notna(x) else "")

display(paper_coef)

save_df(paper_coef, "revised_table_06_paper_style_key_coefficients_long.csv")

pivot_coef = paper_coef.pivot_table(
    index="variable",
    columns="model",
    values="coef_stars",
    aggfunc="first"
)

display(pivot_coef)
pivot_coef.to_csv(OUT_DIR / "revised_table_07_paper_style_key_coefficients_wide.csv", encoding="utf-8-sig")

,model,variable,coef,std_err,t_or_z,p_value,nobs,stars,coef_stars,std_err_fmt
125,M1_controls,LogAssets_w,0.021954,0.005623,3.904039,9.460051e-05,4610.0,***,0.0220***,(0.0056)
126,M1_controls,Leverage_w,0.102597,0.045088,2.275489,2.287662e-02,4610.0,**,0.1026**,(0.0451)
127,M1_controls,ROA_w,-0.363758,0.137095,-2.653319,7.970457e-03,4610.0,***,-0.3638***,(0.1371)
128,M1_controls,Loss,0.013719,0.021696,0.632351,5.271575e-01,4610.0,,0.0137,(0.0217)
253,M2_governance,LogAssets_w,0.028878,0.007833,3.686910,2.269936e-04,3218.0,***,0.0289***,(0.0078)
254,M2_governance,Leverage_w,0.060294,0.049976,1.206463,2.276391e-01,3218.0,,0.0603,(0.0500)
255,M2_governance,ROA_w,-0.294410,0.155249,-1.896368,5.791136e-02,3218.0,*,-0.2944*,(0.1552)
256,M2_governance,Loss,0.033132,0.025597,1.294339,1.955483e-01,3218.0,,0.0331,(0.0256)
257,M2_governance,OutsideDirectorRatio_w,0.070696,0.062767,1.126322,2.600292e-01,3218.0,,0.0707,(0.0628)
258,M2_governance,LargestShareholderOwnership_w,-0.001866,0.000558,-3.346445,8.185496e-04,3218.0,***,-0.0019***,(0.0006)


Saved: c:\Users\starw\.vscode\practice\output_governance_direction\revised_table_06_paper_style_key_coefficients_long.csv | shape=(60, 10)


model,M1_controls,M2_governance,M3_governance_auditor_basic,M5_material_controls,M6_material_governance,M7_material_auditor_basic,P1_count_controls,P2_count_governance,P3_count_auditor_basic
variable,,,,,,,,,
Big4Auditor,NaN,NaN,-0.0280,NaN,NaN,-0.0062,NaN,NaN,nan
CleanOpinion,NaN,NaN,0.0132,NaN,NaN,0.0216,NaN,NaN,nan
LargestShareholderOwnership_w,NaN,-0.0019***,-0.0018***,NaN,-0.0014***,-0.0014***,NaN,nan,nan
Leverage_w,0.1026**,0.0603,0.0600,-0.0032,-0.0214,-0.0230,0.5631***,nan,nan
LogAssets_w,0.0220***,0.0289***,0.0320***,0.0165***,0.0231***,0.0239***,0.1088***,nan,nan
LogMinorityShareholderCount_w,NaN,-0.0075,-0.0059,NaN,-0.0105,-0.0104,NaN,nan,nan
Loss,0.0137,0.0331,0.0319,0.0084,0.0150,0.0144,0.0820,nan,nan
OutsideDirectorRatio_w,NaN,0.0707,0.0761,NaN,0.0937**,0.0960**,NaN,nan,nan
ROA_w,-0.3638***,-0.2944*,-0.2923*,-0.1825**,-0.2035*,-0.2093*,-2.6754***,nan,nan


In [29]:
# ============================================================
# Revised Cell F. Final copy-paste summary for ChatGPT
# ============================================================

compact = {
    "research_direction": {
        "main_title_candidate": "Disclosure Corrections as a Signal of Reporting Quality: Governance, Ownership Structure, and Auditor Characteristics",
        "main_focus": "Governance and ownership structure",
        "auxiliary_focus": "Auditor characteristics and audit fee/non-audit fee variables",
        "main_note": "MinorityShareholderRatio was dropped because it is entirely missing; LogMinorityShareholderCount is used instead."
    },
    "sample_info": {
        "pilot_mode": PILOT_MODE,
        "number_of_firms": int(final_panel["corp_code"].nunique()),
        "number_of_firm_years": int(len(final_panel)),
        "start_year": int(final_panel["year"].min()),
        "end_year": int(final_panel["year"].max()),
        "number_of_sectors": int(final_panel["sector"].nunique())
    },
    "variable_summary": {},
    "missing_rates": {},
    "yearly_summary": yearly.to_dict(orient="records") if "yearly" in globals() else [],
    "regression_sample_sizes": reg_sample_table.to_dict(orient="records") if "reg_sample_table" in globals() else [],
    "regression_key_coefficients": [],
    "paper_style_coefficients": []
}

summary_vars = [
    "CorrectionDummy", "CorrectionCount",
    "MaterialCorrectionDummy", "MaterialCorrectionCount",
    "OutsideDirectorRatio", "LargestShareholderOwnership",
    "MinorityShareholderCount", "LogMinorityShareholderCount",
    "LogAssets", "Leverage", "ROA", "Loss",
    "Big4Auditor", "CleanOpinion",
    "NonAuditFeeRatio", "AuditFee", "NonAuditFee"
]

for var in summary_vars:
    if var in reg.columns:
        s = reg[var]
        compact["missing_rates"][var] = float(s.isna().mean())
        
        if pd.api.types.is_numeric_dtype(s):
            compact["variable_summary"][var] = {
                "non_missing": int(s.notna().sum()),
                "mean": None if pd.isna(s.mean()) else float(s.mean()),
                "median": None if pd.isna(s.median()) else float(s.median()),
                "std": None if pd.isna(s.std()) else float(s.std()),
                "min": None if pd.isna(s.min()) else float(s.min()),
                "max": None if pd.isna(s.max()) else float(s.max())
            }

if "coef_table" in globals() and not coef_table.empty:
    important_terms = [
        "OutsideDirectorRatio_w",
        "LargestShareholderOwnership_w",
        "LogMinorityShareholderCount_w",
        "LogAssets_w",
        "Leverage_w",
        "ROA_w",
        "Loss",
        "Big4Auditor",
        "CleanOpinion",
        "NonAuditFeeRatio_w",
        "LogAuditFee_w",
        "LogNonAuditFee_w"
    ]
    
    compact["regression_key_coefficients"] = (
        coef_table[coef_table["variable"].isin(important_terms)]
        .to_dict(orient="records")
    )

if "paper_coef" in globals() and not paper_coef.empty:
    compact["paper_style_coefficients"] = paper_coef.to_dict(orient="records")

print("\n" + "=" * 80)
print("COMPACT COPY-PASTE BLOCK FOR CHATGPT")
print("=" * 80)
print(json.dumps(compact, ensure_ascii=False, indent=2))
print("=" * 80)
print("END OF FINAL RESULT SUMMARY")
print("=" * 80)

with open(OUT_DIR / "revised_copy_paste_summary_for_chatgpt.json", "w", encoding="utf-8") as f:
    json.dump(compact, f, ensure_ascii=False, indent=2)

print("\nSaved files:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)


COMPACT COPY-PASTE BLOCK FOR CHATGPT
{
  "research_direction": {
    "main_title_candidate": "Disclosure Corrections as a Signal of Reporting Quality: Governance, Ownership Structure, and Auditor Characteristics",
    "main_focus": "Governance and ownership structure",
    "auxiliary_focus": "Auditor characteristics and audit fee/non-audit fee variables",
    "main_note": "MinorityShareholderRatio was dropped because it is entirely missing; LogMinorityShareholderCount is used instead."
  },
  "sample_info": {
    "pilot_mode": false,
    "number_of_firms": 711,
    "number_of_firm_years": 4977,
    "start_year": 2018,
    "end_year": 2024,
    "number_of_sectors": 119
  },
  "variable_summary": {
    "CorrectionDummy": {
      "non_missing": 4977,
      "mean": 0.19931685754470566,
      "median": 0.0,
      "std": 0.39952686983202484,
      "min": 0.0,
      "max": 1.0
    },
    "CorrectionCount": {
      "non_missing": 4977,
      "mean": 0.2943540285312437,
      "median": 0.0,
  